<a href="https://colab.research.google.com/github/DangHuuLong/Ai-Recruiter-Mini-Ai-Service/blob/experiment%2Ffine-tune-similarity-colab-v0.3/notebooks/fine_tune_similarity_v0.3_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune CV-JD Similarity Model — v0.3

**Model:** `sentence-transformers/all-MiniLM-L6-v2`

**Dataset:** 7,000 CV-JD pairs (v0.3) — 4,900 train / 1,050 val / 1,050 test

**Mục tiêu:** 20 epochs, best checkpoint tự động (load_best_model_at_end)

---

## Kết quả các phiên bản trước (test set)

| Phiên bản              | MAE    | RMSE   | Label Accuracy |
|------------------------|--------|--------|----------------|
| Baseline (không fine-tune) | 20.32  | 25.50  | 32.29%        |
| Fine-tuned v0.2 (3 epochs) | 11.92  | 17.50  | 47.71%        |
| **Fine-tuned v0.3 (target)** | **?** | **?** | **?**         |

> Baseline đo trên 350 pairs từ dataset v0.2 test split.
> Fine-tuned v0.2 đo trên cùng test split đó sau 3 epochs.


In [1]:
!pip install sentence-transformers datasets torch

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 1. Clone Repository

Clone repo từ GitHub và checkout branch thực nghiệm v0.3.
Mỗi lần Colab runtime khởi động lại, thư mục /content/ bị xóa — cell này sẽ tự bỏ qua nếu repo đã tồn tại.


In [11]:
import os

if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service.git /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/fine-tune-similarity-colab-v0.3
!git pull origin experiment/fine-tune-similarity-colab-v0.3

/content/Ai-Recruiter-Mini-Ai-Service
Already on 'experiment/fine-tune-similarity-colab-v0.3'
Your branch is up to date with 'origin/experiment/fine-tune-similarity-colab-v0.3'.
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 12 (delta 6), reused 12 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 597.28 KiB | 3.49 MiB/s, done.
From https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service
 * branch            experiment/fine-tune-similarity-colab-v0.3 -> FETCH_HEAD
   1f91725..9dc28ae  experiment/fine-tune-similarity-colab-v0.3 -> origin/experiment/fine-tune-similarity-colab-v0.3
Updating 1f91725..9dc28ae
Fast-forward
 datasets/versions/v0.3/cv_jd_pairs.jsonl      | 7000 +++++++++++++++++++++++++
 datasets/versions/v0.3/job_descriptions.jsonl | 1400 +++++
 datasets/versions/v0.3/resumes.jsonl          | 1400 +++++
 training/check_fine_tune_readiness.py         |    2 +

In [9]:
REPO_PATH = "/content/Ai-Recruiter-Mini-Ai-Service"
DATASET_VERSION = "v0.3"

## 2. Load Dataset
Đọc 3 file JSONL từ dataset v0.3 và kiểm tra phân bổ split/label.


In [12]:
import json
import sys
from pathlib import Path

sys.path.insert(0, REPO_PATH)

def read_jsonl(path):
    records = []
    for line in Path(path).read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if line:
            records.append(json.loads(line))
    return records

VERSION_PATH = f"{REPO_PATH}/datasets/versions/{DATASET_VERSION}"
pairs   = read_jsonl(f"{VERSION_PATH}/cv_jd_pairs.jsonl")
resumes = read_jsonl(f"{VERSION_PATH}/resumes.jsonl")
jds     = read_jsonl(f"{VERSION_PATH}/job_descriptions.jsonl")

# Split distribution
from collections import Counter
split_counts = Counter(p["split"] for p in pairs)
label_counts = Counter(p["label"] for p in pairs)

print(f"Total pairs : {len(pairs)}")
print(f"Resumes     : {len(resumes)}")
print(f"JDs         : {len(jds)}")
print()
print("Split distribution:")
for split in ["train", "validation", "test"]:
    print(f"  {split:12s}: {split_counts[split]}")
print()
print("Label distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  {label:20s}: {count}")


Total pairs : 7000
Resumes     : 1400
JDs         : 1400

Split distribution:
  train       : 4900
  validation  : 1050
  test        : 1050

Label distribution:
  excellent_match     : 1400
  moderate_match      : 1400
  poor_match          : 1400
  strong_match        : 1400
  weak_match          : 1400


## 3. Training Configuration
Chạy 20 epochs với best checkpoint tự động — sau mỗi epoch evaluate trên
validation set, chỉ giữ lại model tốt nhất (theo Spearman cosine correlation).


In [13]:
BASE_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR    = f"{REPO_PATH}/artifacts/models/fine-tuned-miniLM-v0.3"
REPORT_PATH   = f"{REPO_PATH}/artifacts/reports/fine_tune_v0.3_report.json"
EPOCHS        = 20
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
WARMUP_RATIO  = 0.1
SEED          = 42

print(f"Base model  : {BASE_MODEL}")
print(f"Epochs      : {EPOCHS}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Output dir  : {OUTPUT_DIR}")


Base model  : sentence-transformers/all-MiniLM-L6-v2
Epochs      : 20
Batch size  : 16
Output dir  : /content/Ai-Recruiter-Mini-Ai-Service/artifacts/models/fine-tuned-miniLM-v0.3


## 4. Build Training Data
Chuyển pairs thành InputExample cho SentenceTransformer.
Label được normalize về [0, 1] (overall_score / 100).


In [14]:
resume_by_id = {str(r["id"]): r for r in resumes}
jd_by_id     = {str(j["id"]): j for j in jds}

def build_text(resume):
    return "\n".join(filter(None, [
        f"Summary: {resume.get('summary', '')}",
        f"Skills: {', '.join(resume.get('skills', []))}",
        f"Experience: {resume.get('experience_years', '')} years",
    ]))

def build_jd_text(jd):
    return "\n".join(filter(None, [
        f"Title: {jd.get('title', '')}",
        f"Level: {jd.get('level', '')}",
        f"Required skills: {', '.join(jd.get('required_skills', []))}",
        f"Requirements: {'; '.join(jd.get('requirements', []))}",
    ]))

train_pairs = [p for p in pairs if p["split"] == "train"]
val_pairs   = [p for p in pairs if p["split"] == "validation"]
test_pairs  = [p for p in pairs if p["split"] == "test"]

from datasets import Dataset as HFDataset

def to_hf_dataset(pair_list):
    return HFDataset.from_dict({
        "sentence1": [build_text(resume_by_id[str(p["resume_id"])]) for p in pair_list],
        "sentence2": [build_jd_text(jd_by_id[str(p["job_description_id"])]) for p in pair_list],
        "label":     [p["overall_score"] / 100.0 for p in pair_list],
    })

train_dataset = to_hf_dataset(train_pairs)
val_dataset   = to_hf_dataset(val_pairs)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_pairs)}")


Train: 4900 | Val: 1050 | Test: 1050
